# Script 2 — MNIST: load, clean, and visualize

**MNIST** is a set of 70,000 scanned handwritten digits, 0 through 9, written by US
census workers and high-school students. It is the standard "hello world" of image
classification.

**How the images are stored.** Each digit is a 28x28 grayscale picture — 784 pixels
total. But the dataset does not hand you a 28x28 square. It hands you those 784 pixels
*unrolled into a single flat row*, read left-to-right then top-to-bottom, like reading a
page. Each value is one pixel's brightness from 0 (black background) to 255 (white ink).

So the data is a table of 70,000 rows x 784 columns of numbers, and the "image" only
reappears when we fold a row back into a 28x28 grid.

Here we keep only the **3s and 8s** — a deliberately hard pair, since both are built
from stacked curves and differ mainly in whether the left side is closed.

In [ ]:
# Step 1: load MNIST into a pandas DataFrame.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cs156.data import mnist
from cs156.plotting import show_grid

# First call downloads ~15MB from OpenML and caches it in ~/scikit_learn_data,
# so this is slow once and instant every time after.
X, y = mnist()   # X: (70000, 784) pixel brightnesses, y: (70000,) the true digit

# Name the columns pixel_0 ... pixel_783 so the flat row order is explicit.
pixel_cols = [f"pixel_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=pixel_cols)
df["label"] = y   # the digit a human says each image shows

print("rows, columns:", df.shape)      # (70000, 785) = 784 pixels + 1 label
print("pixel range:", X.min(), "-", X.max())   # 0 = black, 255 = white
df["label"].value_counts().sort_index()

### Step 2: clean — drop every digit except 3 and 8

Same boolean-masking idea as the Iris script. `.isin([3, 8])` gives one True/False per
image, and indexing with it keeps only the rows that were True.

In [ ]:
# Step 2: keep only the 3s and the 8s.
keep = df["label"].isin([3, 8])   # True/False for each of the 70,000 images
pair = df[keep].copy()

print(f"kept {len(pair):,} of {len(df):,} images ({len(pair) / len(df):.1%})")
print()
print(pair["label"].value_counts().sort_index())

# Split the table back into a pixel block and a label block for plotting.
pair_pixels = pair[pixel_cols].to_numpy()   # (n, 784)
pair_labels = pair["label"].to_numpy()      # (n,)

### Step 3: fold the flat rows back into pictures

`show_grid` (in `src/cs156/plotting.py`) draws a row of small images. It expects real
28x28 squares, so we have to undo the flattening first.

`.reshape(-1, 28, 28)` does exactly that: it re-cuts each 784-long row into 28 rows of
28 pixels. The `-1` means "figure out how many images that is." `imshow` inside the
helper then paints each number as a shade of gray — 0 black, 255 white.

In [ ]:
# Step 3: show a sample of digits so we can see the handwriting vary.
rng = np.random.default_rng(0)   # fixed seed so the same digits appear each run
sample = rng.choice(len(pair_pixels), size=24, replace=False)

# Unroll -> re-roll: (24, 784) flat rows become (24, 28, 28) actual images.
images = pair_pixels[sample].reshape(-1, 28, 28)
labels = pair_labels[sample]

fig = show_grid(images, labels, cols=8, figsize=(10, 4.5))
fig.suptitle("MNIST 3s and 8s — title above each image is the true label", y=1.02)
plt.show()

### The same digit, many hands

The grid above mixes the two classes. It is more instructive to put all the 3s in one
row and all the 8s in another, because then the variation you are looking at is
*handwriting style* rather than *which digit it is*.

In [ ]:
# Step 3b: one row of 3s, one row of 8s, to isolate the handwriting variation.
for digit in (3, 8):
    rows = pair_pixels[pair_labels == digit]
    picks = rng.choice(len(rows), size=8, replace=False)
    fig = show_grid(rows[picks].reshape(-1, 28, 28), cols=8, figsize=(10, 1.6))
    fig.suptitle(f"eight different people writing a {digit}", y=1.15)
    plt.show()

**What to notice.** Every image is the same 28x28 grid at the same scale, yet no two
strokes are alike: some 8s are drawn as two stacked circles, others as a single crossing
loop; some 3s are round, others nearly angular. Slant, stroke thickness, and where the
digit sits in the frame all move around.

That variation is the entire problem. A rule you might write by hand — "an 8 has two
closed loops" — breaks the moment someone leaves a loop slightly open. This is why the
task is handed to a model that learns the pattern from 13,966 examples instead.